In [ ]:
# --- UNIVERSAL BOOTSTRAP CELL (Har notebook ke shuru mein istemal karein) ---
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# Sahi library versions install karein jo aapas mein theek kaam karte hain
%pip -q install --upgrade --force-reinstall \
  "numpy==2.1.3" "pandas==2.2.2" "scipy==1.14.1" "scikit-learn==1.5.2" \
  "joblib==1.4.2" "pyyaml" "yfinance" "pandas-ta"

# AHEM: Naye software ko load karne ke liye runtime ko khud-ba-khud restart karein
import os
print("Sahi libraries install ho gayi hain. Ab unhein load karne ke liye runtime restart kiya ja raha hai...")
os.kill(os.getpid(), 9)

Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.8/60.8 kB 5.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 949.2/949.2 kB 22.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Requested yfinance from https://files.pythonhosted.org/packages/f3/23/0e28fa29eba03f33d74c58296f301064930340622be34b008ed02d4486de/yfinance-0.1.91-py2.py3-none-any.whl has invalid metadata: Expected matching RIGHT_PARENTHESIS for LEFT_PARENTHESIS, after version specifier
    appdirs (>=1.4.4cryptography>=3.3.2)
            ~~~~~~~~~~^
Please use pip<24.1 if you need to use this version.
Requested yfinance from https://files.pythonhosted.org/packages/f3/23/0e28fa29eba03f33d74c58296f301064930340622be34b008ed02d4486de/yfinance-0.1.91-py2.py3-none-any.whl

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

ROOT = "/content/drive/MyDrive/ARVisionGold"
!pip -q install yfinance pandas pandas-ta scikit-learn joblib pyyaml

import os, yaml, joblib, pandas as pd, numpy as np
import yfinance as yf
import pandas_ta as ta

with open(f"{ROOT}/configs/paths.yaml") as f:
    P = yaml.safe_load(f)
with open(f"{ROOT}/configs/signals.yaml") as f:
    SIG = yaml.safe_load(f)
with open(f"{ROOT}/configs/model.yaml") as f:
    MCFG = yaml.safe_load(f)

MODEL_PATH = P["artifacts"]["model_rf"]
CANDLE_TXT = P["artifacts"]["candle_type"]

rf = joblib.load(MODEL_PATH)
feature_cols = MCFG["features"]["include"]
print("Loaded model ✔")


Mounted at /content/drive
Loaded model ✔


In [4]:
# --- Helper Functions ko dobara define karein (Behtar Version) ---

def compute_indicators(df):
    # Step 1: Column names ko hamesha theek karein
    df.columns = [str(col).capitalize() for col in df.columns]

    # Step 2: Har indicator ko ahtiyat se calculate karein
    out = pd.DataFrame(index=df.index)
    out["Open"]=df["Open"]; out["High"]=df["High"]; out["Low"]=df["Low"]; out["Close"]=df["Close"]; out["Volume"]=df["Volume"].fillna(0)

    # RSI
    rsi = ta.rsi(close=df["Close"], length=14)
    if rsi is not None:
        out["RSI"] = rsi

    # MACD
    macd = ta.macd(close=df["Close"], fast=12, slow=26, signal=9)
    if macd is not None and not macd.empty:
        out["MACD"]=macd.iloc[:,0]
        out["MACD_signal"]=macd.iloc[:,1]
        out["MACD_hist"]=macd.iloc[:,2]

    # Bollinger Bands
    bb = ta.bbands(close=df["Close"], length=20, std=2)
    if bb is not None and not bb.empty:
        out["BB_lower"]=bb.filter(like="BBL").iloc[:,0]
        out["BB_middle"]=bb.filter(like="BBM").iloc[:,0]
        out["BB_upper"]=bb.filter(like="BBU").iloc[:,0]

    # ATR
    atr = ta.atr(high=df["High"], low=df["Low"], close=df["Close"], length=14)
    if atr is not None:
        out["ATR"]=atr if not isinstance(atr, pd.DataFrame) else atr.iloc[:,0]

    return out.dropna()

def generate_signal(model_pred_label:str, candle_type:str, sig_cfg:dict):
    bull_words = set(map(str.upper, sig_cfg["fusion"]["bullish_words"]))
    bear_words = set(map(str.upper, sig_cfg["fusion"]["bearish_words"]))
    model_pred_upper = (model_pred_label or "").upper()
    candle_type_upper = (candle_type or "").upper()

    if model_pred_upper in bull_words and candle_type_upper == 'BULLISH':
        return 'STRONG BUY'
    elif model_pred_upper in bear_words and candle_type_upper == 'BEARISH':
        return 'STRONG SELL'
    elif model_pred_upper in bull_words and candle_type_upper == 'BEARISH':
        return 'WAIT - CONFLICTING SIGNALS (Model is Bullish, Candle is Bearish)'
    elif model_pred_upper in bear_words and candle_type_upper == 'BULLISH':
        return 'WAIT - CONFLICTING SIGNALS (Model is Bearish, Candle is Bullish)'
    return 'UNKNOWN'

print("Tamam zaroori functions ka behtar version is notebook mein load ho gaya hai. ✅")

Tamam zaroori functions ka behtar version is notebook mein load ho gaya hai. ✅


In [7]:
# --- Final Integration Logic (Sab se Behtar Version) ---
try:
    print("--- Step 1: Trained AI model ko load kiya ja raha hai... ---")
    model = joblib.load(MODEL_PATH)
    print("Model kamyabi se load ho gaya.")

    print("\n--- Step 2: AI prediction ke liye taza market data hasil kiya ja raha hai... ---")
    # --- FIX: 1 mahinay ke bajaye 3 mahinay ka data maangein taake indicators fail na hon ---
    live_df = yf.download("XAUUSD=X", period="3mo", interval="1d", progress=False)
    if live_df.empty:
        print("XAUUSD=X se data nahi mila, GC=F try kiya ja raha hai...")
        live_df = yf.download("GC=F", period="3mo", interval="1d", progress=False)
    # --------------------------------------------------------------------------------------

    features_df = compute_indicators(live_df)
    latest_features = features_df.iloc[[-1]] # Aakhri row

    model_features = MCFG["features"]["include"]

    # Yaqeen karein ke taza data mein wohi columns hain jo model ko chahiye
    final_features_for_model = [f for f in model_features if f in latest_features.columns]
    latest_features = latest_features[final_features_for_model]

    prediction_code = model.predict(latest_features)[0]
    predicted_trend = "BULLISH" if prediction_code == 1 else "BEARISH"
    print(f"AI Model ki taza Prediction: {predicted_trend}")

    print("\n--- Step 3: Chart (CV Module) ka natija load kiya ja raha hai... ---")
    # Hum pehle se banayi hui candle_type.txt istemal kar rahe hain
    with open(CANDLE_TXT, 'r') as f:
        latest_candle_type = f.read().strip()
    print(f"Chart per Aakhri Candle: {latest_candle_type}")

    print("\n--- Step 4: Dono natijon ko mila kar Aakhri Faisla... ---")
    final_trading_signal = generate_signal(
        model_pred_label=predicted_trend,
        candle_type=latest_candle_type,
        sig_cfg=SIG_CFG
    )

    print("\n" + "="*40)
    print("          FINAL TRADING RECOMMENDATION")
    print("="*40)
    print(f"\n            --->   {final_trading_signal}   <---")
    print("\n" + "="*40)

except Exception as e:
    print(f"\nEk Error aa gaya hai: {e}")

--- Step 1: Trained AI model ko load kiya ja raha hai... ---
Model kamyabi se load ho gaya.

--- Step 2: AI prediction ke liye taza market data hasil kiya ja raha hai... ---


/tmp/ipython-input-3330153615.py:9: FutureWarning: YF.download() has changed argument auto_adjust default to True
  live_df = yf.download("XAUUSD=X", period="3mo", interval="1d", progress=False)
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['XAUUSD=X']: YFPricesMissingError('possibly delisted; no price data found  (period=3mo) (Yahoo error = "No data found, symbol may be delisted")')


XAUUSD=X se data nahi mila, GC=F try kiya ja raha hai...

Ek Error aa gaya hai: 'Open'


/tmp/ipython-input-3330153615.py:12: FutureWarning: YF.download() has changed argument auto_adjust default to True
  live_df = yf.download("GC=F", period="3mo", interval="1d", progress=False)


In [8]:
import pandas as pd, numpy as np, yfinance as yf

# --- Normalize OHLC columns to single level + proper case ---
def _normalize_ohlc_columns(df: pd.DataFrame) -> pd.DataFrame:
    # Flatten MultiIndex if present
    if isinstance(df.columns, pd.MultiIndex):
        new_cols = []
        for col in df.columns:
            if isinstance(col, tuple):
                # Prefer field names if present
                pick = None
                for part in col:
                    if str(part) in ["Open","High","Low","Close","Adj Close","Volume"]:
                        pick = str(part); break
                new_cols.append(pick if pick else "_".join([str(x) for x in col if x]))
            else:
                new_cols.append(str(col))
        df.columns = new_cols

    # Standardize case
    mapping = {
        "open":"Open", "high":"High", "low":"Low", "close":"Close",
        "adj close":"Adj Close", "volume":"Volume"
    }
    df.columns = [mapping.get(str(c).lower(), str(c)) for c in df.columns]

    # Some tickers miss Volume; keep a column anyway
    if "Volume" not in df.columns:
        df["Volume"] = 0.0
    return df

# --- Yahoo fetch with fallbacks and correct params ---
def fetch_ohlc_with_fallback(
    symbols=("GC=F", "XAUUSD=X", "XAU=X"),
    period="1y",                 # >= 6mo so MACD/BB/ATR compute without NaNs
    interval="1d",
) -> tuple[pd.DataFrame, str]:
    last_err = None
    for sym in symbols:
        try:
            df = yf.download(
                sym, period=period, interval=interval,
                progress=False, group_by="column", auto_adjust=False  # IMPORTANT
            )
            if df is not None and not df.empty:
                df.index.name = "Date"
                df = df.dropna(how="all")
                df = _normalize_ohlc_columns(df)
                if {"Open","High","Low","Close"}.issubset(df.columns):
                    return df, sym
        except Exception as e:
            last_err = e
    raise RuntimeError(f"Yahoo fetch failed for {symbols}. Last error: {last_err}")


In [9]:
import pandas_ta as ta
import joblib, yaml, os, pandas as pd, numpy as np

# Load model & configs already done earlier in your notebook:
# rf = joblib.load(MODEL_PATH)
# feature_cols = MCFG["features"]["include"]

def compute_indicators_exact(df: pd.DataFrame) -> pd.DataFrame:
    """Same recipe as Module-1 (RSI, MACD, BB, ATR)"""
    rsi  = ta.rsi(close=df["Close"], length=14)
    macd = ta.macd(close=df["Close"], fast=12, slow=26, signal=9)  # 3 cols
    bb   = ta.bbands(close=df["Close"], length=20, std=2)          # BBL/BBM/BBU
    atr  = ta.atr(high=df["High"], low=df["Low"], close=df["Close"], length=14)

    out = pd.DataFrame(index=df.index)
    out["Open"]=df["Open"]; out["High"]=df["High"]; out["Low"]=df["Low"]; out["Close"]=df["Close"]
    out["Volume"]=df.get("Volume", 0).fillna(0)

    out["RSI"]         = rsi
    out["MACD"]        = macd.iloc[:, 0]   # MACD
    out["MACD_signal"] = macd.iloc[:, 1]   # MACDs
    out["MACD_hist"]   = macd.iloc[:, 2]   # MACDh

    out["BB_lower"]    = bb.filter(like="BBL").iloc[:, 0]
    out["BB_middle"]   = bb.filter(like="BBM").iloc[:, 0]
    out["BB_upper"]    = bb.filter(like="BBU").iloc[:, 0]

    out["ATR"]         = atr if not isinstance(atr, pd.DataFrame) else atr.iloc[:, 0]
    return out

# 1) FETCH enough history + normalize columns
live_df, used_symbol = fetch_ohlc_with_fallback(
    symbols=("GC=F","XAUUSD=X","XAU=X"),
    period="1y",         # 1y ensures MACD/BB/ATR stable
    interval="1d"
)
print("Using symbol:", used_symbol, "| Rows:", len(live_df))

# 2) Compute indicators and drop warmup NaNs
feats_full = compute_indicators_exact(live_df).dropna()
assert not feats_full.empty, "Indicators empty; increase period to '2y' or check data."

# 3) Pick the latest row and align to model's training feature list exactly
latest = feats_full.iloc[[-1]].copy()
required = list(getattr(rf, "feature_names_in_", feature_cols))  # exact names from training
missing = [c for c in required if c not in latest.columns]
if missing:
    # Shouldn't happen now, but if it does, it's safer to fill with last valid/0 to avoid crash
    for m in missing:
        latest[m] = np.nan
    latest = latest.fillna(method="ffill").fillna(method="bfill").fillna(0)
    print("WARNING: Missing features filled:", missing)

X_live = latest[required]
print("Live feature vector columns:", list(X_live.columns))


Using symbol: GC=F | Rows: 253
Live feature vector columns: ['Open', 'High', 'Low', 'Close', 'Volume', 'RSI', 'MACD', 'MACD_signal', 'MACD_hist', 'BB_upper', 'BB_middle', 'BB_lower', 'ATR']


In [10]:
# Model prediction
label_idx = int(rf.predict(X_live)[0])
label = "UP" if label_idx == 1 else "DOWN"
prob  = float(rf.predict_proba(X_live)[0].max()) if hasattr(rf, "predict_proba") else 0.5

# Candle type (Module-2)
def load_candle_type(path):
    try:
        with open(path) as f:
            return f.read().strip().upper()
    except FileNotFoundError:
        return None

ctype = load_candle_type(CANDLE_TXT)

# Same fusion as Module-3
def fuse_signal(model_label:str, model_prob:float, candle_type:str, cfg:dict):
    mlabel=(model_label or "").upper(); ctype=(candle_type or "").upper()
    bull=set(map(str.upper, cfg["fusion"]["bullish_words"]))
    bear=set(map(str.upper, cfg["fusion"]["bearish_words"]))
    th=cfg["thresholds"]
    up=mlabel in bull; dn=mlabel in bear
    ct_bull=(ctype=="BULLISH"); ct_bear=(ctype=="BEARISH")
    if up and model_prob>=th["rf_prob_buy"] and ct_bull: return "STRONG BUY"
    if dn and model_prob>=th["rf_prob_sell"] and ct_bear: return "STRONG SELL"
    if model_prob<th["no_trade_band"]: return "WAIT"
    if up: return "BUY"
    if dn: return "SELL"
    return "WAIT"

signal = fuse_signal(label, prob, ctype, SIG)
print(f"Symbol: {used_symbol} | Model: {label} (p≈{prob:.2f}) | Candle: {ctype or 'N/A'} | Final Signal: {signal}")


Symbol: GC=F | Model: DOWN (p≈0.61) | Candle: BULLISH | Final Signal: SELL
